# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SafeeAkmal/FlyRank-ML-internship-tasks/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Ans: A page is flagged CTR_UNDERPERFORM when its actual CTR sits meaningfully below
the average CTR of pages at the same position tier, because position is the
single biggest driver of CTR, comparing raw CTR across different positions is
unfair; you have to compare within tier.

A page is flagged LOW_VOLUME_UNRELIABLE when it doesn't have enough impressions
to trust any CTR comparison at all. A page with 40 impressions swinging from
0% to 10% CTR isn't a signal, it's noise.

Everything else is NO_FLAG, not because it's fine, just because this rule
doesn't have enough evidence to say anything about it.

Reason codes this rule can output:
- CTR_BELOW_TIER_EXPECTATION
- LOW_VOLUME_INSUFFICIENT_DATA
- NO_FLAG

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
from google.colab import userdata
hf_token = userdata.get('hf_token')
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}')")

MONTH = "2026-03"
fact_path    = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/data_0.parquet"
content_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

df = con.execute(f"""
    WITH agg AS (
        SELECT content_hash_id, client_hash_id,
            SUM(gsc_impressions) FILTER (WHERE gsc_data_available) AS impressions_30d,
            SUM(gsc_clicks)      FILTER (WHERE gsc_data_available) AS clicks_30d,
            SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS avg_position_30d
        FROM read_parquet('{fact_path}')
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT a.*, c.is_published, c.is_deleted
    FROM agg a
    JOIN read_parquet('{content_path}') c USING (content_hash_id)
    WHERE c.is_published = TRUE AND c.is_deleted = FALSE
      AND a.impressions_30d > 0
""").df()

df["ctr_30d"] = df["clicks_30d"] / df["impressions_30d"]

def position_bucket(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"
df["position_bucket"] = df["avg_position_30d"].apply(position_bucket)

signal1 = df.groupby("position_bucket").agg(
    n=("ctr_30d", "size"),
    avg_ctr=("ctr_30d", "mean")
).reindex(["1-3","4-10","11-20","21+"])

print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                     n   avg_ctr
position_bucket                 
1-3              18840  0.011706
4-10             83158  0.004878
11-20            29914  0.003284
21+              44656  0.001953


In [2]:
# Verdict cell — read the table above and decide
# Expectation: avg_ctr should DECREASE as position bucket gets worse.
print("VERDICT: CONFIRMED" if signal1["avg_ctr"].is_monotonic_decreasing else "VERDICT: check — not monotonic, may be MIXED")

VERDICT: CONFIRMED


In [4]:
def volume_bucket(imp):
    if imp < 200: return "low (<200)"
    if imp < 1000: return "mid (200-1000)"
    return "high (1000+)"
df["volume_bucket"] = df["impressions_30d"].apply(volume_bucket)

signal2 = df.groupby("volume_bucket").agg(
    n=("ctr_30d", "size"),
    avg_ctr=("ctr_30d", "mean"),
    ctr_std=("ctr_30d", "std")
).reindex(["low (<200)","mid (200-1000)","high (1000+)"])

print(signal2)
# Verdict: quick-win logic assumes low-volume pages have noisier (higher variance) CTR
# — that's the thing to confirm before trusting any CTR comparison on them.
print("VERDICT: CONFIRMED" if signal2["ctr_std"].iloc[0] > signal2["ctr_std"].iloc[-1] else "VERDICT: OPPOSITE — low volume is NOT noisier than expected")

                    n   avg_ctr   ctr_std
volume_bucket                            
low (<200)      91759  0.006370  0.052212
mid (200-1000)  39757  0.002369  0.004208
high (1000+)    45052  0.002950  0.003420
VERDICT: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

MIN_IMPRESSIONS = 200  # from signal 2's verdict — below this, CTR is unreliable

def score_row(row):
    tier_avg = signal1.loc[row["position_bucket"], "avg_ctr"]
    gap = tier_avg - row["ctr_30d"]  # positive = underperforming its tier

    if row["impressions_30d"] < MIN_IMPRESSIONS:
        return pd.Series({"score": 0.0, "reason_code": "LOW_VOLUME_INSUFFICIENT_DATA", "action": "NO_ACTION"})
    if gap > 0.01:  # underperforming tier by more than 1 pt of CTR
        return pd.Series({"score": gap * row["impressions_30d"], "reason_code": "CTR_BELOW_TIER_EXPECTATION", "action": "REVIEW_METADATA"})
    return pd.Series({"score": 0.0, "reason_code": "NO_FLAG", "action": "NO_ACTION"})

df[["score", "reason_code", "action"]] = df.apply(score_row, axis=1)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

out_cols = ["rank","content_hash_id","client_hash_id","impressions_30d","clicks_30d",
            "ctr_30d","position_bucket","score","reason_code","action"]

import os
os.makedirs("work/outputs", exist_ok=True)
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked[out_cols].head(10))

   rank           content_hash_id           client_hash_id  impressions_30d  \
0     1  content_44f34c0a90047651  client_23a62021009f63c4         212404.0   
1     2  content_8d7d99f109e19aa2  client_e547b89c05043229         203497.0   
2     3  content_8e1334d6356668e3  client_73cda7b4e4f265ea         134984.0   
3     4  content_fec55986a1868d62  client_73cda7b4e4f265ea         124075.0   
4     5  content_9c057b66c30a3abb  client_73cda7b4e4f265ea          83834.0   
5     6  content_9ef3d7516483e665  client_e547b89c05043229          89229.0   
6     7  content_306bc78dff1eb683  client_e547b89c05043229          80821.0   
7     8  content_c46df0fa61530d86  client_e547b89c05043229          70398.0   
8     9  content_6a9c79f55413b447  client_73cda7b4e4f265ea          73272.0   
9    10  content_252aa5480bb1f8d7  client_73cda7b4e4f265ea          66698.0   

   clicks_30d   ctr_30d position_bucket        score  \
0        24.0  0.000113             1-3  2462.370215   
1       289.0  0.0

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Ans:
## Top-20 review

Every row in this queue carries reason_code = CTR_BELOW_TIER_EXPECTATION and
action = REVIEW_METADATA, but this review revealed that the classification is
driven by a data-quality defect (see Section 4), not genuine underperformance.
Confidence notes below reflect that finding.

| Rank | Reason code (as scored) | Confidence | What would make it wrong | How you'd know |
|---|---|---|---|---|
| 1-20 | CTR_BELOW_TIER_EXPECTATION | Very low — see Section 4 | Position 1-3 classification is driven by a handful of days with gsc_avg_position < 1.0 (an invalid value), not sustained top-3 ranking | Traced directly on content_44f34c0a90047651: normal position 5-18 on 25/31 days, then 6 days at position 0.08-0.24 accounting for 93% of impressions |
| 5 (content_9c057b66c30a3abb) | CTR_BELOW_TIER_EXPECTATION | Very low — confirmed defect case | 6 of 31 days impossible-position, covering 99.9% of this page's impressions | Directly identified in the dataset-wide impossible-position scan (Section 4) |
| 3, 4 (client_73cda7b4e4f265ea, 1-click rows) | CTR_BELOW_TIER_EXPECTATION | Very low | Single-click counts against six-figure impressions are implausible without a tracking bug | Same client appears 6x in this top 20 — cross-check this client's raw GSC export against the warehouse |
| 18 (0 clicks, 44,707 impressions) | CTR_BELOW_TIER_EXPECTATION | Very low | Zero clicks at genuine position 1-3 is essentially impossible | Same defect pattern as above — verify via daily breakdown |

No row in this top 20 should be treated as an actionable REVIEW_METADATA item
until the underlying position computation is corrected — see Section 4 for
why.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked[out_cols].head(20)
print(top20.to_string(index=False))

 rank          content_hash_id          client_hash_id  impressions_30d  clicks_30d  ctr_30d position_bucket       score                reason_code          action
    1 content_44f34c0a90047651 client_23a62021009f63c4         212404.0        24.0 0.000113             1-3 2462.370215 CTR_BELOW_TIER_EXPECTATION REVIEW_METADATA
    2 content_8d7d99f109e19aa2 client_e547b89c05043229         203497.0       289.0 0.001420             1-3 2093.106173 CTR_BELOW_TIER_EXPECTATION REVIEW_METADATA
    3 content_8e1334d6356668e3 client_73cda7b4e4f265ea         134984.0         1.0 0.000007             1-3 1579.102998 CTR_BELOW_TIER_EXPECTATION REVIEW_METADATA
    4 content_fec55986a1868d62 client_73cda7b4e4f265ea         124075.0         1.0 0.000008             1-3 1451.403836 CTR_BELOW_TIER_EXPECTATION REVIEW_METADATA
    5 content_9c057b66c30a3abb client_73cda7b4e4f265ea          83834.0         1.0 0.000012             1-3  980.348565 CTR_BELOW_TIER_EXPECTATION REVIEW_METADATA
    6 content_9e

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Ans: Root cause: a subset of daily rows in fact_content_daily_performance report
gsc_avg_position < 1.0, which is not a valid Search Console position (rank
starts at 1). Scope: 68,425 of 176,738 content items (38.7%) show at least
one such day in March 2026. Restricting to pages with real volume (>=500
monthly impressions), 15,117 items (8.5%) are affected, and these rows
account for 2.5% of all March impression volume in the warehouse.

Two distinct sub-patterns, confirmed by splitting on n_impossible_days:
- **Burst** (15,089 pages, 99.8% of affected material pages): a handful of
  anomalous days within an otherwise normal month. Average 2,394 total
  impressions, with only 15.6% of impressions on average coming from the
  impossible-position days. Traced directly on content_44f34c0a90047651:
  normal position 5-18 on 25 of 31 days, then a 6-day burst (Mar 24-29) of
  10,000s of daily impressions at position 0.08-0.24, clicks flat at 0-2 —
  consistent with bot/crawler traffic or a dated attribution glitch.
- **Always impossible** (28 pages): position invalid on nearly every day of
  the month (avg 94.2% of impressions on impossible-position days). Likely a
  different mechanism — e.g. a non-standard SERP feature (rich snippet, image
  pack) being mis-encoded as position <1 for the entire period, rather than a
  transient error.

Confirmed not client-specific: excluding the 3 highest-volume flagged clients
from the original top-20, position-1-3 CTR median remained 0.0% (n=10,485).
content_9c057b66c30a3abb, ranked #5 in the original baseline queue (83,834
impressions, 1 click), independently appears in this scan as a burst case
(6 impossible-position days covering 99.9% of its monthly impressions) —
confirming the top-20 anomalies and this systemic pattern are the same
underlying defect.

Leakage check: the score uses only impressions_30d, clicks_30d, ctr_30d,
avg_position_30d, and position_bucket — all observed within the March window,
none derived from a future period or a FlyRank product flag. No
health_score/priority_score/action_type/refresh_tier columns exist in this
dataframe.

Conclusion: the baseline rule should not be used to generate real action items
on any page touched by this defect until gsc_avg_position < 1 rows are
filtered or corrected upstream. Recommended immediate mitigation: exclude days
with gsc_avg_position < 1 from the aggregation, and route any page where such
days account for a majority of its impressions to a manual data-quality
check rather than a metadata-review action. This affects an identifiable,
quantified share of the dataset (8.5% of material pages, 2.5% of impression
volume) — it is a data-quality defect, not a rule-tuning problem, and no
threshold change in this baseline fixes it.However,this was diagnosed on March 2026 only; whether this is a persistent pipeline defect or a March-specific incident is unconfirmed and should be checked across the full 18-month history before any cross-month comparison work begins.

In [14]:
# 1. Per-client concentration check
suspect = ranked[ranked["position_bucket"] == "1-3"]
by_client = suspect.groupby("client_hash_id").agg(
    n_flagged=("score", "size"),
    avg_ctr=("ctr_30d", "mean"),
    avg_impressions=("impressions_30d", "mean")
).sort_values("n_flagged", ascending=False)
print(by_client.head(10))

# 2. Overall position-1-3 CTR distribution (not just top 20)
print(df[df["position_bucket"] == "1-3"]["ctr_30d"].describe())

# 3. Confirm not isolated to top 3 suspect clients
suspect_clients = ["client_73cda7b4e4f265ea", "client_62f4a7e64f5e0096", "client_e547b89c05043229"]
clean_slice = df[(df["position_bucket"] == "1-3") & (~df["client_hash_id"].isin(suspect_clients))]
print(clean_slice["ctr_30d"].describe())

# 4. Trace the mechanism on one page's raw daily rows
suspect_id = "content_44f34c0a90047651"
daily = con.execute(f"""
    SELECT report_date, gsc_impressions, gsc_clicks, gsc_avg_position, gsc_sum_position, gsc_data_available
    FROM read_parquet('{fact_path}')
    WHERE content_hash_id = '{suspect_id}'
    ORDER BY report_date
""").df()
print(daily.to_string(index=False))

# 5. Dataset-wide scan for the impossible-position defect
daily_all = con.execute(f"""
    SELECT content_hash_id, client_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
    FROM read_parquet('{fact_path}')
    WHERE gsc_data_available = TRUE
""").df()
daily_all["impossible_position"] = daily_all["gsc_avg_position"] < 1.0

impossible_summary = daily_all.groupby("content_hash_id").agg(
    n_impossible_days=("impossible_position", "sum"),
    impressions_on_impossible_days=("gsc_impressions", lambda s: s[daily_all.loc[s.index, "impossible_position"]].sum()),
    total_impressions=("gsc_impressions", "sum")
)
impossible_summary["pct_impressions_from_impossible_days"] = (
    impossible_summary["impressions_on_impossible_days"] / impossible_summary["total_impressions"]
)
affected = impossible_summary[impossible_summary["n_impossible_days"] > 0]

material = affected[affected["total_impressions"] >= 500].sort_values(
    "pct_impressions_from_impossible_days", ascending=False
)

total_impressions_all = daily_all["gsc_impressions"].sum()
total_impressions_impossible = daily_all.loc[daily_all["impossible_position"], "gsc_impressions"].sum()

print(f"Pages with any impossible-position day: {len(affected)} of {len(impossible_summary)}")
print(f"Pages with >=500 impressions affected: {len(material)}")
print(f"Share of ALL March impressions from impossible-position rows: {total_impressions_impossible/total_impressions_all:.1%}")

# 6. Split into the two sub-patterns
material["pattern"] = material["n_impossible_days"].apply(
    lambda n: "always_impossible" if n >= 28 else "burst"
)
print(material["pattern"].value_counts())
print(material.groupby("pattern")[["total_impressions", "pct_impressions_from_impossible_days"]].mean())

# 7. Leakage check — confirm the score used only allowed inputs
used_cols = ["impressions_30d", "clicks_30d", "ctr_30d", "avg_position_30d", "position_bucket"]
forbidden = ["health_score", "priority_score", "action_type", "refresh_tier"]
print("Forbidden columns present in df?", any(c in df.columns for c in forbidden))
print("Score built only from:", used_cols)

                         n_flagged   avg_ctr  avg_impressions
client_hash_id                                               
client_73cda7b4e4f265ea       4280  0.003378      4029.996963
client_62f4a7e64f5e0096       3407  0.002744      3870.604638
client_3ffa76342f366962       2950  0.047369         4.607119
client_a80fca3f171ed1de       1471  0.005191       869.446635
client_08a6a72ff48e62c0        806  0.003063        99.437965
client_fef1a8f436438636        748  0.002595      1371.085561
client_23a62021009f63c4        688  0.002434      1300.527616
client_e547b89c05043229        668  0.002602      6765.127246
client_0fa64a184f18a4a0        411  0.006248      1476.437956
client_3f0ce4d44fe94f3d        395  0.003189      1159.372152
count    18840.000000
mean         0.011706
std          0.082125
min          0.000000
25%          0.000000
50%          0.000000
75%          0.002890
max          1.000000
Name: ctr_30d, dtype: float64
count    10485.000000
mean         0.018598
std   

In [15]:
# Run the impossible-position scan across every available month
months = ["2025-01","2025-02","2025-03","2025-04","2025-05","2025-06",
          "2025-07","2025-08","2025-09","2025-10","2025-11","2025-12",
          "2026-01","2026-02","2026-03","2026-04","2026-05","2026-06"]

monthly_results = []
for m in months:
    path = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}/data_0.parquet"
    row = con.execute(f"""
        SELECT
            '{m}' AS month,
            COUNT(*) AS total_rows,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_impressions) FILTER (WHERE gsc_avg_position < 1.0) AS impressions_impossible
        FROM read_parquet('{path}')
        WHERE gsc_data_available = TRUE
    """).df()
    monthly_results.append(row)

trend = pd.concat(monthly_results, ignore_index=True)
trend["pct_impossible"] = trend["impressions_impossible"] / trend["total_impressions"]
print(trend[["month","pct_impossible"]])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      month  pct_impossible
0   2025-01        0.000433
1   2025-02        0.000454
2   2025-03        0.000188
3   2025-04        0.002268
4   2025-05        0.001962
5   2025-06        0.001131
6   2025-07        0.008669
7   2025-08        0.015818
8   2025-09        0.045515
9   2025-10        0.047196
10  2025-11        0.037479
11  2025-12        0.037423
12  2026-01        0.039171
13  2026-02        0.043202
14  2026-03        0.025229
15  2026-04        0.014631
16  2026-05        0.002506
17  2026-06        0.003241


## Cross-month validation

The impossible-position defect (gsc_avg_position < 1.0) was checked across
all 18 available months rather than treated as a March-only finding:

| Period | pct_impossible |
|---|---|
| 2025-01 to 2025-06 | 0.02% - 0.2% (near-negligible) |
| 2025-07 to 2025-08 | rising: 0.9% - 1.6% |
| 2025-09 to 2026-02 | elevated plateau: 3.7% - 4.7% (peak: Oct 2025) |
| 2026-03 to 2026-06 | declining: 2.5% -> 0.3% |

This is a bounded episode, not a persistent background bug: it emerged
around July 2025, peaked in September-October 2025, and has been recovering
since, nearly returning to baseline by June 2026. The March 2026 figure used
in this baseline (2.5%) sits on the declining tail — meaningfully lower than
the September-October 2025 peak (~4.7%) and should not be read as
representative of the defect's worst impact.

Implication: any historical ranking, trend, or client comparison that spans
July 2025 through February 2026 is at elevated risk of the same distortion
demonstrated here, and should not be trusted without the same
gsc_avg_position < 1 filter applied. This also suggests a traceable root
cause with a start and (near) end date — worth escalating to whoever owns
the GSC ingestion pipeline to identify what changed around July 2025 and
what changed around March 2026, since narrowing to those two windows is far
more tractable than debugging "the pipeline is sometimes wrong."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.